# Signal Classification — Training & Evaluation

This notebook demonstrates how the signal classifier works and how to evaluate/improve it.

**Approach**: We use a hybrid classification strategy:
1. **Embedding similarity**: Compare article embeddings to signal type descriptions
2. **Keyword patterns**: Regex-based pattern matching for high-precision signals
3. **Zero-shot classification**: Optional transformers pipeline for nuanced cases

The final score blends all available methods for robust classification.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
from app.agents.classifier_agent import ClassifierAgent, SIGNAL_TYPE_LABELS, KEYWORD_PATTERNS
from app.agents.embedding_agent import EmbeddingAgent

classifier = ClassifierAgent()
embedder = EmbeddingAgent()

print('Agents loaded successfully')

## 1. Labeled Test Dataset

We define a labeled dataset of article headlines with known signal types for evaluation.

In [ ]:
TEST_DATA = [
    # Funding signals
    ('Tabby raises USD 200M Series D at USD 3.5B valuation', 'funding'),
    ('INVIA secures USD 1.2M seed funding for SME financial OS', 'funding'),
    ('G42 secures USD 1.5B investment from Microsoft', 'funding'),
    ('Sarwa raises USD 15M Series B for GCC expansion', 'funding'),
    
    # Expansion signals
    ('Bain Capital opens Abu Dhabi office for MENA deals', 'expansion'),
    ('Stripe secures DIFC license, opens Dubai office', 'expansion'),
    ('Databricks opens Dubai MENA headquarters', 'expansion'),
    ('Kitopi opens 15 new cloud kitchens across Saudi Arabia', 'expansion'),
    
    # Partnership signals
    ('G42 and Microsoft expand sovereign cloud partnership', 'partnership'),
    ('Mubadala partners with G42 on AI infrastructure', 'partnership'),
    ('Swvl partners with Dubai RTA on smart routes', 'partnership'),
    
    # Launch signals
    ('Anghami launches AI Arabic music recommendation engine', 'launch'),
    ('Emirates Steel begins green steel production', 'launch'),
    ('Careem launches digital wallet and P2P payments', 'launch'),
    
    # M&A signals
    ('IHC completes USD 1.1B acquisition of Turkish healthcare group', 'm_and_a'),
    ('Edafa acquires waste recycling startup Cyclex', 'm_and_a'),
    
    # Regulatory signals
    ('Revolut receives DIFC Innovation License for UAE', 'regulatory'),
    ('Rain obtains full VARA license for Dubai crypto trading', 'regulatory'),
    
    # Non-signals (should score low)
    ('Manchester United signs new sponsorship deal', 'none'),
    ('Weather forecast for Dubai shows sunny skies ahead', 'none'),
]

print(f'Test dataset: {len(TEST_DATA)} examples')
print(f'Signal types: {set(t for _, t in TEST_DATA)}')

## 2. Evaluate Classifier Accuracy

In [ ]:
correct = 0
total = 0
results = []

print(f'{"Headline":<55} {"Expected":<12} {"Predicted":<12} {"Conf":>5} {"OK?":>4}')
print('=' * 95)

for headline, expected in TEST_DATA:
    result = classifier.classify(headline)
    predicted = result.signal_type if result.confidence >= 0.1 else 'none'
    is_correct = predicted == expected
    if expected != 'none':
        correct += int(is_correct)
        total += 1
    elif expected == 'none' and result.confidence < 0.15:
        correct += 1
        total += 1
    else:
        total += 1
    
    mark = 'OK' if is_correct else 'MISS'
    print(f'{headline[:53]:<55} {expected:<12} {predicted:<12} {result.confidence:>4.0%} {mark:>4}')
    results.append((headline, expected, predicted, result.confidence))

accuracy = correct / total if total > 0 else 0
print(f'\nAccuracy: {correct}/{total} = {accuracy:.1%}')

## 3. Embedding Space Visualization

Visualize how articles cluster in the embedding space by signal type.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    
    # Embed all test headlines
    texts = [h for h, _ in TEST_DATA]
    labels = [t for _, t in TEST_DATA]
    embeddings = embedder.encode_batch(texts)
    
    # Reduce to 2D with PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeddings)
    
    # Plot
    colors = {
        'funding': '#2E7D5B', 'expansion': '#1B4F72', 'partnership': '#7B2D8E',
        'launch': '#C77A1F', 'm_and_a': '#B33A3A', 'regulatory': '#0E1E3F',
        'hiring': '#4F6691', 'executive': '#9A7843', 'none': '#CCCCCC',
    }
    
    fig, ax = plt.subplots(figsize=(10, 7))
    for label in set(labels):
        mask = [l == label for l in labels]
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            label=label, c=colors.get(label, '#999'),
            s=100, alpha=0.8, edgecolors='white', linewidth=0.5,
        )
    
    ax.set_title('Article Embeddings by Signal Type (PCA)', fontsize=14)
    ax.legend(loc='best', framealpha=0.9)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.tight_layout()
    plt.show()
    print('Explained variance:', pca.explained_variance_ratio_.round(3))
    
except ImportError:
    print('matplotlib/sklearn not available. Install with: pip install matplotlib scikit-learn')

## 4. Confidence Distribution Analysis

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    signal_confs = [c for _, e, _, c in results if e != 'none']
    noise_confs = [c for _, e, _, c in results if e == 'none']
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(signal_confs, bins=10, alpha=0.7, label='Signal articles', color='#1B4F72')
    ax.hist(noise_confs, bins=10, alpha=0.7, label='Non-signal articles', color='#CCCCCC')
    ax.axvline(x=0.15, color='red', linestyle='--', label='Threshold (0.15)')
    ax.set_xlabel('Confidence Score')
    ax.set_ylabel('Count')
    ax.set_title('Classification Confidence Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('matplotlib not available for visualization')

## 5. Keyword Pattern Coverage Analysis

In [ ]:
import re

print('=== Keyword Pattern Coverage ===')
print(f'{"Signal Type":<15} {"Patterns":>8} {"Test Matches":>12}')
print('-' * 40)

for signal_type, patterns in KEYWORD_PATTERNS.items():
    matches = 0
    for headline, expected in TEST_DATA:
        if expected == signal_type:
            text_lower = headline.lower()
            if any(re.search(p, text_lower) for p in patterns):
                matches += 1
    expected_count = sum(1 for _, e in TEST_DATA if e == signal_type)
    print(f'{signal_type:<15} {len(patterns):>8} {matches}/{expected_count:>10}')

## Summary

The hybrid classification approach provides:
- **High precision** from keyword patterns (zero false positives)
- **High recall** from embeddings (catches semantically similar articles)
- **Graceful degradation** to keyword-only mode when ML models are unavailable

### Next Steps for Improvement:
1. Expand the labeled dataset with more diverse examples
2. Fine-tune a small classifier on domain-specific data
3. Add active learning to improve from pipeline feedback
4. Implement confidence calibration for better threshold tuning